In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 加载 LFW 数据集
lfw = fetch_lfw_people(data_home="/storage/data/zhanghx2023", min_faces_per_person=20, resize=0.4)
X = lfw.data      # (n_samples, h*w)
y = lfw.target    # (n_samples,)
num_classes = len(lfw.target_names)
h, w = lfw.images.shape[1], lfw.images.shape[2]
print(f"Loaded LFW: {X.shape[0]} samples, image size {h}x{w}, {num_classes} classes")

# 3. 数据标准化和划分
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 转为 Tensor 并重塑为图像
X_train = torch.tensor(X_train, dtype=torch.float32).view(-1, 1, h, w).to(device)
X_test  = torch.tensor(X_test,  dtype=torch.float32).view(-1, 1, h, w).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_test  = torch.tensor(y_test,  dtype=torch.long).to(device)

# 5. 数据集和加载器
class LFWDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

batch_size = 32
train_loader = DataLoader(LFWDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(LFWDataset(X_test,  y_test),  batch_size=batch_size, shuffle=False)

# 6. 模型定义
class AlexNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5), nn.Linear(384, 256), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # 简化版 ResNet18
        def conv3x3(in_c, out_c, stride=1): return nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        class Block(nn.Module):
            def __init__(self, in_c, out_c, stride=1):
                super().__init__()
                self.conv1 = conv3x3(in_c, out_c, stride)
                self.bn1 = nn.BatchNorm2d(out_c)
                self.conv2 = conv3x3(out_c, out_c)
                self.bn2 = nn.BatchNorm2d(out_c)
                self.shortcut = nn.Sequential()
                if stride!=1 or in_c!=out_c:
                    self.shortcut = nn.Sequential(
                        nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                        nn.BatchNorm2d(out_c)
                    )
            def forward(self, x):
                out = nn.ReLU()(self.bn1(self.conv1(x)))
                out = self.bn2(self.conv2(out)) + self.shortcut(x)
                return nn.ReLU()(out)
        self.layer1 = Block(1, 32, stride=2)
        self.layer2 = Block(32, 64, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(64, num_classes)
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128,3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(), nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

# class MLP(nn.Module):
#     def __init__(self, num_classes, input_dim):
#         super().__init__()
#         self.fc = nn.Sequential(
#             nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.5),
#             nn.Linear(256, num_classes)
#         )
#     def forward(self, x): return self.fc(x.view(x.size(0), -1))

# MLP
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # 展平输入
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# 7. 训练与评估函数
def train(model, loader, criterion, optimizer, epochs=20):
    model.to(device).train()
    for ep in range(epochs):
        total_loss = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {ep+1}/{epochs}, Loss={total_loss/len(loader):.4f}")

def evaluate(model, loader):
    model.to(device).eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds==yb).sum().item()
            total += yb.size(0)
    print(f"Accuracy: {100*correct/total:.2f}%")


Loaded LFW: 3023 samples, image size 50x37, 62 classes


## AlexNet训练评估

In [28]:
print(f"\n=== Training AlexNet ===")
m = AlexNet(num_classes)
optimzer = optim.Adam(m.parameters(), lr=0.0005)
criterion= nn.CrossEntropyLoss()
train(m, train_loader, criterion, optimzer, epochs=200)
print(f"--- Evaluating AlexNet ---")
evaluate(m, test_loader)


=== Training AlexNet ===
Epoch 1/200, Loss=3.8190
Epoch 2/200, Loss=3.7073
Epoch 3/200, Loss=3.6492
Epoch 4/200, Loss=3.6201
Epoch 5/200, Loss=3.5919
Epoch 6/200, Loss=3.5617
Epoch 7/200, Loss=3.5287
Epoch 8/200, Loss=3.5001
Epoch 9/200, Loss=3.4576
Epoch 10/200, Loss=3.3941
Epoch 11/200, Loss=3.3372
Epoch 12/200, Loss=3.2998
Epoch 13/200, Loss=3.2412
Epoch 14/200, Loss=3.1794
Epoch 15/200, Loss=3.1410
Epoch 16/200, Loss=3.1191
Epoch 17/200, Loss=3.0450
Epoch 18/200, Loss=3.0326
Epoch 19/200, Loss=2.9879
Epoch 20/200, Loss=2.9192
Epoch 21/200, Loss=2.9078
Epoch 22/200, Loss=2.8470
Epoch 23/200, Loss=2.7736
Epoch 24/200, Loss=2.7145
Epoch 25/200, Loss=2.6389
Epoch 26/200, Loss=2.5862
Epoch 27/200, Loss=2.5844
Epoch 28/200, Loss=2.4843
Epoch 29/200, Loss=2.4263
Epoch 30/200, Loss=2.4089
Epoch 31/200, Loss=2.3479
Epoch 32/200, Loss=2.2833
Epoch 33/200, Loss=2.2314
Epoch 34/200, Loss=2.1892
Epoch 35/200, Loss=2.1198
Epoch 36/200, Loss=2.0494
Epoch 37/200, Loss=2.0200
Epoch 38/200, Loss=1.

## ResNet

In [ ]:
print(f"\n=== Training ResNet ===")
m = ResNet18(num_classes)
optimzer = optim.Adam(m.parameters(), lr=0.001,weight_decay=1e-4)
criterion= nn.CrossEntropyLoss()
train(m, train_loader, criterion, optimzer, epochs=150)
print(f"--- Evaluating ResNet ---")
evaluate(m, test_loader)


=== Training ResNet ===
Epoch 1/100, Loss=3.7138
Epoch 2/100, Loss=3.4916
Epoch 3/100, Loss=3.3866
Epoch 4/100, Loss=3.2522
Epoch 5/100, Loss=3.1022
Epoch 6/100, Loss=2.9669
Epoch 7/100, Loss=2.8393
Epoch 8/100, Loss=2.7031
Epoch 9/100, Loss=2.5662
Epoch 10/100, Loss=2.4594
Epoch 11/100, Loss=2.3214
Epoch 12/100, Loss=2.2254
Epoch 13/100, Loss=2.1010
Epoch 14/100, Loss=1.9792
Epoch 15/100, Loss=1.8818
Epoch 16/100, Loss=1.7737
Epoch 17/100, Loss=1.6569
Epoch 18/100, Loss=1.5825
Epoch 19/100, Loss=1.4772
Epoch 20/100, Loss=1.3776
Epoch 21/100, Loss=1.2969
Epoch 22/100, Loss=1.2174
Epoch 23/100, Loss=1.1218
Epoch 24/100, Loss=1.0283
Epoch 25/100, Loss=0.9723
Epoch 26/100, Loss=0.8894
Epoch 27/100, Loss=0.8297
Epoch 28/100, Loss=0.7707
Epoch 29/100, Loss=0.7065
Epoch 30/100, Loss=0.6355
Epoch 31/100, Loss=0.6011
Epoch 32/100, Loss=0.5449
Epoch 33/100, Loss=0.4967
Epoch 34/100, Loss=0.4704
Epoch 35/100, Loss=0.4065
Epoch 36/100, Loss=0.3766
Epoch 37/100, Loss=0.3448
Epoch 38/100, Loss=0.3

## Small CNN

In [26]:
print(f"\n=== Training SmallCNN ===")
m = CustomCNN(num_classes)
optimzer = optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
criterion= nn.CrossEntropyLoss()
train(m, train_loader, criterion, optimzer, epochs=150)
print(f"--- Evaluating SmallCNN ---")
evaluate(m, test_loader)


=== Training SmallCNN ===
Epoch 1/150, Loss=3.7432
Epoch 2/150, Loss=3.6067
Epoch 3/150, Loss=3.5613
Epoch 4/150, Loss=3.5315
Epoch 5/150, Loss=3.5287
Epoch 6/150, Loss=3.5100
Epoch 7/150, Loss=3.4993
Epoch 8/150, Loss=3.4875
Epoch 9/150, Loss=3.4747
Epoch 10/150, Loss=3.4690
Epoch 11/150, Loss=3.4222
Epoch 12/150, Loss=3.3710
Epoch 13/150, Loss=3.3170
Epoch 14/150, Loss=3.2852
Epoch 15/150, Loss=3.2516
Epoch 16/150, Loss=3.2152
Epoch 17/150, Loss=3.1911
Epoch 18/150, Loss=3.1596
Epoch 19/150, Loss=3.1314
Epoch 20/150, Loss=3.1246
Epoch 21/150, Loss=3.0923
Epoch 22/150, Loss=3.0530
Epoch 23/150, Loss=3.0218
Epoch 24/150, Loss=2.9914
Epoch 25/150, Loss=2.9279
Epoch 26/150, Loss=2.9067
Epoch 27/150, Loss=2.8634
Epoch 28/150, Loss=2.8316
Epoch 29/150, Loss=2.7680
Epoch 30/150, Loss=2.7286
Epoch 31/150, Loss=2.6886
Epoch 32/150, Loss=2.6447
Epoch 33/150, Loss=2.5940
Epoch 34/150, Loss=2.5639
Epoch 35/150, Loss=2.5341
Epoch 36/150, Loss=2.4790
Epoch 37/150, Loss=2.4584
Epoch 38/150, Loss=2

In [25]:
print(f"\n=== Training MLP ===")
m = MLP(num_classes=num_classes,hidden_size=256,input_size=h*w)
# m=MLP(num_classes, input_dim=h*w)
optimzer = optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-3)
criterion= nn.CrossEntropyLoss()
train(m, train_loader, criterion, optimzer, epochs=150)
print(f"--- Evaluating MLP ---")
evaluate(m, test_loader)


=== Training MLP ===
Epoch 1/150, Loss=3.4110
Epoch 2/150, Loss=2.3619
Epoch 3/150, Loss=1.8397
Epoch 4/150, Loss=1.5270
Epoch 5/150, Loss=1.3356
Epoch 6/150, Loss=1.1979
Epoch 7/150, Loss=1.0617
Epoch 8/150, Loss=0.9729
Epoch 9/150, Loss=0.8947
Epoch 10/150, Loss=0.7947
Epoch 11/150, Loss=0.7329
Epoch 12/150, Loss=0.6987
Epoch 13/150, Loss=0.6654
Epoch 14/150, Loss=0.6292
Epoch 15/150, Loss=0.5615
Epoch 16/150, Loss=0.4988
Epoch 17/150, Loss=0.5115
Epoch 18/150, Loss=0.4535
Epoch 19/150, Loss=0.4723
Epoch 20/150, Loss=0.4743
Epoch 21/150, Loss=0.4333
Epoch 22/150, Loss=0.4180
Epoch 23/150, Loss=0.4619
Epoch 24/150, Loss=0.4790
Epoch 25/150, Loss=0.4583
Epoch 26/150, Loss=0.4479
Epoch 27/150, Loss=0.4118
Epoch 28/150, Loss=0.4161
Epoch 29/150, Loss=0.3752
Epoch 30/150, Loss=0.3706
Epoch 31/150, Loss=0.3712
Epoch 32/150, Loss=0.3763
Epoch 33/150, Loss=0.3864
Epoch 34/150, Loss=0.4165
Epoch 35/150, Loss=0.3742
Epoch 36/150, Loss=0.4519
Epoch 37/150, Loss=0.4332
Epoch 38/150, Loss=0.3947